# Satellite Data Acquisition Pipeline
**NB 01** | London, UK | Sentinel-2 | Planetary Computer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX       = (-0.15, 51.47, -0.1, 51.52)
DATE_RANGE = ('2024-06-01', '2024-08-31')
PROVIDERS  = ['planetary_computer', 'element84', 'aws_earth']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 10,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32630', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 7,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'london_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# Spectral indices on clean preprocessed data
if PREPROCESSED.exists():
    idx_dir = DATA_DIR / 'indices'
    idx_dir.mkdir(exist_ok=True)
    results_idx = client.indices.compute_all(
        str(PREPROCESSED),
        indices  = ['ndvi','ndwi','ndbi','mndwi','nbr'],
        output_dir = str(idx_dir),
        band_map = {'blue':1,'green':2,'red':3,'nir':4,'swir1':6,'swir2':7},
    )
    print("Spectral indices computed:")
    import rasterio
    for name, path in results_idx.items():
        with rasterio.open(path) as src:
            a = src.read(1)
        print(f"  {name.upper():<8}: min={a.min():.3f}  max={a.max():.3f}  mean={a.mean():.3f}")


In [ ]:
# Export COG + validation report
if PREPROCESSED.exists():
    COG_OUT = DATA_DIR / 'london_s2_cog.tif'
    client.postprocess.to_cog(str(PREPROCESSED), str(COG_OUT))
    client.validator.generate_report(str(PREPROCESSED), str(DATA_DIR/'validation_report.html'))
    print(f"COG: {COG_OUT}")
    print(f"Validation report: {DATA_DIR}/validation_report.html")


**NB01 — Data Acquisition** complete.
- Study area: London UK
- Sensor: Sentinel-2 L2A
- Model: None (indices only)
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG